In [1]:
import json
import os
from pathlib import Path

import numpy as np
from sentence_transformers import SentenceTransformer

# 1️⃣ Load model — intfloat/e5-base-v2 (768-dim)
model = SentenceTransformer("intfloat/e5-base-v2")

# 2️⃣ Load handbook chunks from scraping.ipynb (list of { "text", "metadata" })
DATA_PROCESSED = Path("data/processed")
with open(DATA_PROCESSED / "academic_handbook_2025_26_chunks.json", "r", encoding="utf-8") as f:
    records = json.load(f)

texts = [r["text"] for r in records]
print(f"✅ Loaded {len(texts)} chunks from academic_handbook_2025_26_chunks.json")

c:\Users\Hasan Jaafar\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Hasan Jaafar\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Hasan Jaafar\.cache\huggingface\hub\models--intfloat--e5-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to 

✅ Loaded 839 chunks from academic_handbook_2025_26_chunks.json


In [2]:
# 3️⃣ Encode all chunk texts
embeddings = model.encode(texts, show_progress_bar=True)
embeddings = np.array(embeddings)
print("✅ Embeddings shape:", embeddings.shape)

# 4️⃣ Save for Meta_data.ipynb / Chroma rebuild
out_dir = Path("data/embeddings")
out_dir.mkdir(parents=True, exist_ok=True)
np.save(out_dir / "academic_handbook_2025_26_e5_base_v2.npy", embeddings)
print("✅ Saved:", (out_dir / "academic_handbook_2025_26_e5_base_v2.npy").resolve())

Batches: 100%|██████████| 27/27 [00:19<00:00,  1.40it/s]

✅ Embeddings shape: (839, 768)
✅ Saved: C:\Users\Hasan Jaafar\Documents\finalyearproject\final submission\chatbot\chatbot\data\embeddings\academic_handbook_2025_26_e5_base_v2.npy


In [3]:
# Build Chroma DB from saved embeddings + chunk JSON (metadata from scraping pipeline)
import json
from pathlib import Path

import numpy as np

DATA_PROCESSED = Path("data/processed")
embeddings = np.load(Path("data/embeddings") / "academic_handbook_2025_26_e5_base_v2.npy")
with open(DATA_PROCESSED / "academic_handbook_2025_26_chunks.json", "r", encoding="utf-8") as f:
    records = json.load(f)

texts = [r["text"] for r in records]


def chroma_metadata(m: dict) -> dict:
    """Chroma accepts str, int, float, bool only."""
    out = {}
    for k, v in m.items():
        if v is None:
            out[k] = ""
        elif isinstance(v, (bool, int, float)):
            out[k] = v
        else:
            out[k] = str(v)
    return out


metadatas = [chroma_metadata(r["metadata"]) for r in records]
print("✅ Example metadata:", metadatas[:3])

✅ Example metadata: [{'document': 'Handbook of Academic Regulations 2025–2026', 'part': '', 'section': '', 'clause': '', 'is_continuation': False}, {'document': 'Handbook of Academic Regulations 2025–2026', 'part': '1', 'section': '1', 'clause': '1.1', 'is_continuation': False}, {'document': 'Handbook of Academic Regulations 2025–2026', 'part': '1', 'section': '1', 'clause': '1.2', 'is_continuation': False}]


In [4]:
# Initialize Chroma & add documents (run after cell 2)
import chromadb
from chromadb.utils import embedding_functions

embed_func = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="intfloat/e5-base-v2"
)

chroma_client = chromadb.PersistentClient(path="data/chroma_db_handbook")

try:
    chroma_client.delete_collection("academic_handbook_2025_26")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="academic_handbook_2025_26",
    embedding_function=embed_func,
)

ids = [f"chunk_{i}" for i in range(len(texts))]
collection.add(
    ids=ids,
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=metadatas,
)

print(f"✅ Added {len(texts)} chunks to Chroma (academic_handbook_2025_26).")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9463.02it/s]
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


✅ Added 839 chunks to Chroma (academic_handbook_2025_26).


In [5]:
query = "mitigating circumstances deadline for academic appeal"
results = collection.query(
    query_texts=[query],
    n_results=3,
    include=["documents", "metadatas", "distances"],
)

for i, (doc, meta, dist) in enumerate(
    zip(results["documents"][0], results["metadatas"][0], results["distances"][0]), 1
):
    sim = 1 - dist
    print(f"\n🔹 Hit {i} | clause {meta.get('clause')} | part {meta.get('part')} | section {meta.get('section')} | sim ≈ {sim:.3f}")
    print(doc[:600])


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



🔹 Hit 1 | clause 11.49 | part 3 | section 11 | sim ≈ 0.788
11.49. Where a student is dissatisfied with the outcome of a mitigating circumstances claim, they have a right to submit an academic appeal. The only ground upon which such an appeal can be
made is that there has been a material irregularity in the conduct of the mitigating
circumstances process (refer to the University’s Academic Appeal Regulations for further
information).

🔹 Hit 2 | clause 16.29 | part 4 | section 16 | sim ≈ 0.786
16.29. Mitigating circumstances will not be considered as grounds for an academic appeal. Any student wishing to have mitigating circumstances considered in respect of an assessment
following the decision of a Progression and Award Board on that assessment should refer to
the University’s Mitigating Circumstances Regulations (Section 11 of the Academic
Regulations).

🔹 Hit 3 | clause 11.20 | part 3 | section 11 | sim ≈ 0.758
11.20. Students with a disability for whom agreed Reasonable Adjustments 